In [ ]:
import os
import gc
import json
import glob
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import timm

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch    : {torch.__version__}")
print(f"torchaudio : {torchaudio.__version__}")
print(f"CUDA       : {torch.cuda.is_available()}")

class CFG:
    seed = 42

    BASE_DIR        = "/kaggle/input/competitions/birdclef-2026"
    TRAIN_AUDIO_DIR = f"{BASE_DIR}/train_audio"
    TRAIN_CSV       = f"{BASE_DIR}/train.csv"
    TAXONOMY_CSV    = f"{BASE_DIR}/taxonomy.csv"
    TEST_DIR        = f"{BASE_DIR}/test_soundscapes"
    SAMPLE_SUB      = f"{BASE_DIR}/sample_submission.csv"
    OUTPUT_DIR      = "/kaggle/working"

    TRAINED_MODEL_PATH = f"{OUTPUT_DIR}/bird_ast_model.pth"

    TARGET_SR    = 32_000
    SEGMENT_SEC  = 5.0

    N_MELS     = 128
    N_FFT      = 1024
    HOP_LENGTH = 320
    FMIN       = 20.0
    FMAX       = 16_000.0

    SPEC_TIME_FRAMES = 512
    PATCH_FREQ       = 16
    PATCH_TIME       = 16

    D_MODEL     = 384
    N_HEADS     = 6
    N_LAYERS    = 12
    MLP_RATIO   = 4.0
    DROPOUT     = 0.1

    N_FOLDS    = 5
    TRAIN_FOLDS= [0, 1, 2, 3]
    VAL_FOLD   = 4
    EPOCHS     = 15
    BATCH_SIZE = 32
    NUM_WORKERS= 2
    LR         = 5e-4
    WEIGHT_DECAY = 1e-4

    DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    AMP_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cfg = CFG()
Path(cfg.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Device : {cfg.DEVICE}")
print(f"AMP    : {cfg.AMP_DEVICE}")

df       = pd.read_csv(cfg.TRAIN_CSV)
taxonomy = pd.read_csv(cfg.TAXONOMY_CSV)

TARGET_COLUMNS = sorted(taxonomy["primary_label"].astype(str).unique().tolist())
NUM_CLASSES    = len(TARGET_COLUMNS)
LABEL2IDX      = {lbl: i for i, lbl in enumerate(TARGET_COLUMNS)}

df["file_path"]     = cfg.TRAIN_AUDIO_DIR + "/" + df["filename"].astype(str)
df["primary_label"] = df["primary_label"].astype(str)

skf = StratifiedKFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.seed)
df["fold"] = -1
for fold_idx, (_, val_idx) in enumerate(skf.split(df, df["primary_label"])):
    df.loc[df.index[val_idx], "fold"] = fold_idx

print(f"Total samples  : {len(df):,}")
print(f"Unique species : {NUM_CLASSES}")
print(df["fold"].value_counts().sort_index())

class BirdCLEFDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        num_classes: int,
        label2idx: dict,
        mode: str = "train",
    ):
        self.df          = df.reset_index(drop=True)
        self.num_classes = num_classes
        self.label2idx   = label2idx
        self.mode        = mode
        self.seg_len     = int(cfg.SEGMENT_SEC * cfg.TARGET_SR)

        self.mel_transform = T.MelSpectrogram(
            sample_rate = cfg.TARGET_SR,
            n_fft       = cfg.N_FFT,
            hop_length  = cfg.HOP_LENGTH,
            n_mels      = cfg.N_MELS,
            f_min       = cfg.FMIN,
            f_max       = cfg.FMAX,
        )
        self.db_transform  = T.AmplitudeToDB(stype="power", top_db=80)
        self.freq_mask     = T.FrequencyMasking(freq_mask_param=15)
        self.time_mask     = T.TimeMasking(time_mask_param=50)

    def __len__(self):
        return len(self.df)

    def _load_wave(self, path: str) -> torch.Tensor:
        try:
            wav, sr = torchaudio.load(path)
        except Exception:
            return torch.zeros(1, self.seg_len)

        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        if wav.shape[1] >= self.seg_len:
            start = random.randint(0, wav.shape[1] - self.seg_len) if self.mode == "train" \
                    else (wav.shape[1] - self.seg_len) // 2
            wav = wav[:, start : start + self.seg_len]
        else:
            wav = F.pad(wav, (0, self.seg_len - wav.shape[1]))

        return wav

    def _to_spec(self, wav: torch.Tensor) -> torch.Tensor:
        spec = self.db_transform(self.mel_transform(wav))
        T_fixed = cfg.SPEC_TIME_FRAMES

        if spec.shape[2] >= T_fixed:
            spec = spec[:, :, :T_fixed]
        else:
            spec = F.pad(spec, (0, T_fixed - spec.shape[2]))

        mean = spec.mean()
        std  = spec.std() + 1e-6
        return (spec - mean) / std

    def __getitem__(self, idx: int):
        row  = self.df.iloc[idx]
        wav  = self._load_wave(row["file_path"])
        spec = self._to_spec(wav)

        if self.mode == "train":
            spec = self.freq_mask(spec)
            spec = self.time_mask(spec)
            if random.random() < 0.3:
                spec = spec + torch.randn_like(spec) * 0.05

        label = torch.zeros(self.num_classes, dtype=torch.float32)
        lbl   = str(row["primary_label"])
        if lbl in self.label2idx:
            label[self.label2idx[lbl]] = 1.0

        return spec, label

class PatchEmbed(nn.Module):
    def __init__(self, freq_bins: int, time_frames: int,
                 patch_freq: int, patch_time: int, d_model: int):
        super().__init__()
        assert freq_bins  % patch_freq == 0
        assert time_frames % patch_time == 0

        self.n_freq  = freq_bins  // patch_freq
        self.n_time  = time_frames // patch_time
        self.n_patch = self.n_freq * self.n_time

        self.proj = nn.Conv2d(
            1, d_model,
            kernel_size=(patch_freq, patch_time),
            stride     =(patch_freq, patch_time),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, mlp_ratio: float, dropout: float):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True
        )
        self.norm2 = nn.LayerNorm(d_model)
        mlp_dim    = int(d_model * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(d_model, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x

class AudioSpectrogramTransformer(nn.Module):
    def __init__(
        self,
        num_classes:  int,
        freq_bins:    int   = CFG.N_MELS,
        time_frames:  int   = CFG.SPEC_TIME_FRAMES,
        patch_freq:   int   = CFG.PATCH_FREQ,
        patch_time:   int   = CFG.PATCH_TIME,
        d_model:      int   = CFG.D_MODEL,
        n_heads:      int   = CFG.N_HEADS,
        n_layers:     int   = CFG.N_LAYERS,
        mlp_ratio:    float = CFG.MLP_RATIO,
        dropout:      float = CFG.DROPOUT,
    ):
        super().__init__()

        self.patch_embed = PatchEmbed(
            freq_bins, time_frames, patch_freq, patch_time, d_model
        )
        n_patch = self.patch_embed.n_patch

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(
            torch.zeros(1, n_patch + 1, d_model)
        )
        self.pos_drop  = nn.Dropout(dropout)

        self.blocks = nn.Sequential(*[
            TransformerBlock(d_model, n_heads, mlp_ratio, dropout)
            for _ in range(n_layers)
        ])

        self.norm = nn.LayerNorm(d_model)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.size(0)

        x = self.patch_embed(x)

        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)

        x = self.pos_drop(x + self.pos_embed)

        x = self.blocks(x)
        x = self.norm(x)

        cls_out = x[:, 0]
        logits  = self.head(cls_out)
        return logits

_dummy = torch.zeros(2, 1, cfg.N_MELS, cfg.SPEC_TIME_FRAMES)
_model = AudioSpectrogramTransformer(num_classes=10)
_out   = _model(_dummy)
print(f"AST sanity check — output shape: {_out.shape}")
del _dummy, _model, _out

class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.float()
        bce     = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs   = torch.sigmoid(logits.float())
        pt      = torch.where(targets == 1, probs, 1 - probs)
        focal_w = self.alpha * (1 - pt) ** self.gamma
        return (focal_w * bce).mean()

def train_one_epoch(model, loader, optimizer, scheduler, criterion, scaler, device):
    model.train()
    running_loss = 0.0

    for specs, labels in tqdm(loader, desc="  train", leave=False):
        specs  = specs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=cfg.AMP_DEVICE):
            preds = model(specs)
            loss  = criterion(preds, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()

    return running_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for specs, labels in tqdm(loader, desc="    val", leave=False):
            specs  = specs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast(device_type=cfg.AMP_DEVICE):
                preds = model(specs)
                loss  = criterion(preds, labels)

            running_loss += loss.item()

    return running_loss / len(loader)

train_df = df[df["fold"].isin(cfg.TRAIN_FOLDS)].reset_index(drop=True)
val_df   = df[df["fold"] == cfg.VAL_FOLD].reset_index(drop=True)

class_counts   = train_df["primary_label"].value_counts().to_dict()
sample_weights = [1.0 / np.sqrt(class_counts[lbl]) for lbl in train_df["primary_label"]]
sampler        = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_dataset = BirdCLEFDataset(train_df, NUM_CLASSES, LABEL2IDX, mode="train")
val_dataset   = BirdCLEFDataset(val_df,   NUM_CLASSES, LABEL2IDX, mode="valid")

train_loader = DataLoader(
    train_dataset, batch_size=cfg.BATCH_SIZE, sampler=sampler,
    num_workers=cfg.NUM_WORKERS, pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False,
    num_workers=cfg.NUM_WORKERS, pin_memory=torch.cuda.is_available(),
)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")

model = AudioSpectrogramTransformer(
    num_classes = NUM_CLASSES,
    freq_bins   = cfg.N_MELS,
    time_frames = cfg.SPEC_TIME_FRAMES,
    patch_freq  = cfg.PATCH_FREQ,
    patch_time  = cfg.PATCH_TIME,
    d_model     = cfg.D_MODEL,
    n_heads     = cfg.N_HEADS,
    n_layers    = cfg.N_LAYERS,
    mlp_ratio   = cfg.MLP_RATIO,
    dropout     = cfg.DROPOUT,
).to(cfg.DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"AST parameters : {n_params:,}")

criterion = FocalLoss(alpha=0.25, gamma=2.0)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY
)

warmup_steps = len(train_loader)
total_steps  = cfg.EPOCHS * len(train_loader)

def lr_lambda(step):
    if step < warmup_steps:
        return float(step) / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
scaler    = torch.amp.GradScaler(device=cfg.AMP_DEVICE)

best_val_loss = float("inf")
history       = []

n_patch = (cfg.N_MELS // cfg.PATCH_FREQ) * (cfg.SPEC_TIME_FRAMES // cfg.PATCH_TIME)
print(f"\nAST training  |  {cfg.DEVICE}  |  {cfg.EPOCHS} epochs  |  {NUM_CLASSES} classes")
print(f"Patches per sample : {n_patch}  ({cfg.N_MELS//cfg.PATCH_FREQ}×{cfg.SPEC_TIME_FRAMES//cfg.PATCH_TIME})")
print("─" * 60)

for epoch in range(1, cfg.EPOCHS + 1):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, scheduler, criterion, scaler, cfg.DEVICE
    )
    val_loss = validate(model, val_loader, criterion, cfg.DEVICE)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

    improved = "*" if val_loss < best_val_loss else ""
    print(
        f"Epoch {epoch:>2}/{cfg.EPOCHS}  "
        f"train={train_loss:.4f}  val={val_loss:.4f}  "
        f"lr={scheduler.get_last_lr()[0]:.2e}  {improved}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "epoch":       epoch,
                "model_state": model.state_dict(),
                "val_loss":    val_loss,
                "num_classes": NUM_CLASSES,
                "cfg_snapshot": {
                    "N_MELS":            cfg.N_MELS,
                    "SPEC_TIME_FRAMES":  cfg.SPEC_TIME_FRAMES,
                    "PATCH_FREQ":        cfg.PATCH_FREQ,
                    "PATCH_TIME":        cfg.PATCH_TIME,
                    "D_MODEL":           cfg.D_MODEL,
                    "N_HEADS":           cfg.N_HEADS,
                    "N_LAYERS":          cfg.N_LAYERS,
                    "MLP_RATIO":         cfg.MLP_RATIO,
                    "DROPOUT":           cfg.DROPOUT,
                },
            },
            cfg.TRAINED_MODEL_PATH,
        )

print(f"\nBest val loss : {best_val_loss:.4f}")
print(f"Checkpoint    : {cfg.TRAINED_MODEL_PATH}")

meta_path = Path(cfg.OUTPUT_DIR) / "target_columns.json"
with open(meta_path, "w") as f:
    json.dump(TARGET_COLUMNS, f)
print(f"Target columns saved : {meta_path}")

print("\nDone.")

import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in history]
train_losses = [h["train_loss"] for h in history]
val_losses = [h["val_loss"] for h in history]

plt.figure(figsize=(10, 6))
plt.plot(epochs, train_losses, label="Train Loss", marker="o", linewidth=2)
plt.plot(epochs, val_losses, label="Validation Loss", marker="s", linewidth=2)
plt.title("AST Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.xticks(epochs)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()

plot_path = Path(cfg.OUTPUT_DIR) / "loss_plot.png"
plt.savefig(plot_path)
print(f"Training plot saved : {plot_path}")
plt.show()



Based on very logs in output, there is a table for better understanding of training process:


Epoch,Train Loss,Val Loss,Learning Rate

1,0.0055,0.0021,5.00e-04

2,0.0022,0.0021,4.94e-04

3,0.0021,0.0020,4.75e-04

4,0.0021,0.0020,4.45e-04

5,0.0021,0.0020,4.06e-04

6,0.0021,0.0020,3.58e-04

7,0.0021,0.0020,3.06e-04

8,0.0021,0.0020,2.50e-04

9,0.0021,0.0020,1.94e-04

10,0.0021,0.0020,1.42e-04

11,0.0020,0.0020,9.41e-05

12,0.0020,0.0020,5.45e-05

13,0.0020,0.0019,2.48e-05

14,0.0020,0.0019,6.27e-06

15,0.0020,0.0019,0.00e+00


The metrics indicate that the model found the pattern of the data very early on:

In Epoch 1, the training loss was 0.0055. By Epoch 2, it plummeted to 0.0022. This suggests that either the learning rate was very effective or the initial weights were already close to a local minimum for this dataset.

Stability: From Epoch 3 to Epoch 15, the loss values became very stable, ending at 0.0020 (train) and 0.0019 (val).


Model: Audio Spectrogram Transformer (AST).
Parameters: 21,611,178 (~21.6M). This is a standard size for an AST-base model, providing a good balance between capacity and training speed.
Environment: PyTorch 2.10.0 with CUDA and AMP (Automatic Mixed Precision) enabled
Data Split: used a 5-fold cross-validation strategy



PyTorch    : 2.10.0+cu128
torchaudio : 2.10.0+cu128
CUDA       : True
Device : cuda
AMP    : cuda
Total samples  : 35,549
Unique species : 234
fold
0    7110
1    7110
2    7110
3    7110
4    7109
Name: count, dtype: int64
AST sanity check — output shape: torch.Size([2, 10])
Train batches : 889
Val   batches : 223
AST parameters : 21,611,178

AST training  |  cuda  |  15 epochs  |  234 classes
Patches per sample : 256  (8×32)
────────────────────────────────────────────────────────────
Epoch  1/15  train=0.0055  val=0.0021  lr=5.00e-04  *
Epoch  2/15  train=0.0022  val=0.0021  lr=4.94e-04  *
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    Exception ignored in: self._shutdown_workers()<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
if w.is_alive():    
self._shutdown_workers() 
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
      if w.is_alive(): 
   ^ ^ ^ ^  ^ ^^^^^^^^^^^^^^^^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^assert self._parent_pid == os.getpid(), 'can only test a child process'
 
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
        assert self._parent_pid == os.getpid(), 'can only test a child process' 
         ^  ^  ^^ ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^AssertionError^: ^^^can only test a child process

AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>assert self._parent_pid == os.getpid(), 'can only test a child process'

 Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
     self._shutdown_workers()  
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
      if w.is_alive(): 
  Exception ignored in:   <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  
Exception ignored in:  ^ Traceback (most recent call last):
^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^ ^
    ^^Traceback (most recent call last):
^self._shutdown_workers()^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^^self._shutdown_workers()    ^if w.is_alive():^
^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^     ^^ if w.is_alive(): ^^
 ^^  ^  ^^   ^^^ ^
 ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^ ^^    assert self._parent_pid == os.getpid(), 'can only test a child process'^^^^^
^^^^^^ ^^ ^^^^ ^^^^^ ^^ ^^^^^ ^
^ 

  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 AssertionError  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
     :     assert self._parent_pid == os.getpid(), 'can only test a child process'can only test a child process assert self._parent_pid == os.getpid(), 'can only test a child process'


  Exception ignored in: ^  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^  ^ 
Traceback (most recent call last):
^    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^       ^  self._shutdown_workers()
 ^  ^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^ ^    ^ if w.is_alive():^^ 
^ ^ ^ ^ ^^^ ^^ ^^^^^ ^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^^^    
^assert self._parent_pid == os.getpid(), 'can only test a child process'^AssertionError^^
^: ^ can only test a child process^^^
^ ^ ^^Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^ ^^
 
Traceback (most recent call last):
^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
AssertionError
 : AssertionError     : self._shutdown_workers()can only test a child process can only test a child process
 

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>    
^Exception ignored in: if w.is_alive():^Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^
 ^Traceback (most recent call last):
     ^ ^ ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 self._shutdown_workers()    ^ self._shutdown_workers()
^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    ^    ^if w.is_alive():^^if w.is_alive():
^
^ ^^  ^ ^  ^ ^ ^^  ^ ^ ^^^ ^^ ^^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^
^^ ^^^ ^^ ^^^ ^^ ^^^ 
^ AssertionError: ^ ^^can only test a child process  

 ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^    Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^assert self._parent_pid == os.getpid(), 'can only test a child process'
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
Traceback (most recent call last):
^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
     ^assert self._parent_pid == os.getpid(), 'can only test a child process'    ^self._shutdown_workers()
  
^ ^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  ^     ^   ^ if w.is_alive():^    
^   ^   ^^ ^^ ^^  ^ ^^^ ^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^AssertionError^^
: ^can only test a child process  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^
    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^Exception ignored in: 
^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^ 
^^ Traceback (most recent call last):
^^^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^ ^     ^^^self._shutdown_workers()
 ^
 AssertionError^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 :  can only test a child process^     
^if w.is_alive():^ Exception ignored in: ^
^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^

 AssertionError^Traceback (most recent call last):
 ^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
: ^ can only test a child process    ^ ^ 
self._shutdown_workers()^^
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^Exception ignored in: ^^    if w.is_alive():^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^

^^Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^^ ^    ^^ ^self._shutdown_workers()^ ^ 
 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ ^
^    ^^if w.is_alive():^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^
^^     ^assert self._parent_pid == os.getpid(), 'can only test a child process'^ 
^^   ^^  ^^  ^ ^ ^^  ^^^^ ^
 
^ AssertionError^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 : ^     can only test a child processassert self._parent_pid == os.getpid(), 'can only test a child process'^^

^^ ^^ Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^ ^
 ^^^Traceback (most recent call last):
 ^^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^ ^    ^ ^ self._shutdown_workers()
^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^        ^^^assert self._parent_pid == os.getpid(), 'can only test a child process'if w.is_alive():^
^
 ^^ ^  ^ ^^^  ^^ ^  ^ ^^ ^ ^^^  ^^ ^^ ^^^^^^ ^^^^^^^^^^^^^^^^^^
^^AssertionError^^: ^^can only test a child process^^^
^^^^^
^^Exception ignored in:   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
^    assert self._parent_pid == os.getpid(), 'can only test a child process'Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^^
^
     AssertionError^self._shutdown_workers(): ^ 
can only test a child process   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^
     ^ if w.is_alive(): Exception ignored in: ^
 ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^ 
 ^ Traceback (most recent call last):
 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
   ^    ^^^ ^self._shutdown_workers()^ 
^^ ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

^    ^AssertionError^if w.is_alive():^: ^^can only test a child process

^^ ^ ^^^ Exception ignored in: ^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^ ^^
 ^Traceback (most recent call last):
 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^^^    ^
self._shutdown_workers()^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^
    ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^assert self._parent_pid == os.getpid(), 'can only test a child process'^    ^
^if w.is_alive(): ^^
  ^^  ^^  ^^  ^^ 
 ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
   ^    ^assert self._parent_pid == os.getpid(), 'can only test a child process' ^  
^ ^ ^^^^ ^^^
^ ^^ AssertionError^^^ : ^^ ^^can only test a child process^^^ 
^^ ^ ^^ ^^^ ^^^^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^    ^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^
^ ^^ ^^ ^^ ^ ^^ ^^^ ^^  ^
 AssertionError^^:  ^^can only test a child process^^
^^^^^^^^^^^^^^^^^^^^^^^^^^
^^AssertionError^^^: ^^can only test a child process^^
^^^^^^^
AssertionError: can only test a child process
Epoch  3/15  train=0.0021  val=0.0020  lr=4.75e-04  *
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
Exception ignored in: Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00><function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  
 
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
        File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
self._shutdown_workers() 
    ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
self._shutdown_workers()^
^      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^if w.is_alive():    
if w.is_alive():^ ^
^  ^  ^ ^ ^    ^ 
  ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^
^^ ^^ ^^ ^ ^^ ^^ ^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^  

^ Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
       File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'     
^self._shutdown_workers() assert self._parent_pid == os.getpid(), 'can only test a child process'^
 ^
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^ ^         if w.is_alive():^^
   ^  ^    ^    ^   ^  ^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^^    ^^^^^^^
assert self._parent_pid == os.getpid(), 'can only test a child process'^^AssertionError^: 
^^ can only test a child process^^ 
^^ ^^^Exception ignored in: ^ ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^^
 ^^^^ Traceback (most recent call last):
^^ ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^ ^

    AssertionErrorAssertionError:  self._shutdown_workers(): can only test a child process can only test a child process
 
^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Exception ignored in: ^Exception ignored in:     ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>if w.is_alive():<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^


^Traceback (most recent call last):
Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    ^ ^self._shutdown_workers()^     
^self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 
^     ^ if w.is_alive():  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^
^ ^    ^ ^if w.is_alive():^ 
^^^  ^^  ^^ ^^  ^ ^^ ^^^^ ^^ ^^^^^^^^^^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^^
^ ^^^ 
^ ^^ AssertionError^ ^: 
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^ can only test a child process    
 ^assert self._parent_pid == os.getpid(), 'can only test a child process' Exception ignored in: ^
 
 <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
    File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
Traceback (most recent call last):
^  ^       File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^ assert self._parent_pid == os.getpid(), 'can only test a child process' 
      self._shutdown_workers()^   ^ 
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^     ^^if w.is_alive(): 
^^ ^ ^ ^ ^ ^ ^^^ ^ ^ ^ ^ ^^ ^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError^^
: can only test a child process ^ ^
^^^^  
Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>AssertionError: ^ can only test a child process
 ^
 Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^     Exception ignored in: self._shutdown_workers()^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 

^ Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
        ^^^self._shutdown_workers()if w.is_alive():
^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^     ^^^ if w.is_alive():
 AssertionError^ 
: ^  ^  can only test a child process  ^ ^
^^^^  ^ Exception ignored in: ^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
^^Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^^^^^    ^self._shutdown_workers()^^

  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^    ^^
if w.is_alive():^ ^
 ^^ ^^  ^  
^  ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
       ^  assert self._parent_pid == os.getpid(), 'can only test a child process' ^ 
 ^ ^  ^ ^^^^^^^^^ ^^^
 ^^AssertionError^^ : ^can only test a child process^^^
  ^
^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
Exception ignored in:     ^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>assert self._parent_pid == os.getpid(), 'can only test a child process' 
^
^Traceback (most recent call last):
^ ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^     ^ ^self._shutdown_workers()^
^ ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  ^    ^^  if w.is_alive():^^
^^ ^ ^^  ^^^  ^ ^^^ ^^^ ^^ ^^^^^^^^^^^^^^
AssertionError^^^: ^^^^can only test a child process^
^^^^^^^^^Exception ignored in: ^^^
^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^Traceback (most recent call last):
^    assert self._parent_pid == os.getpid(), 'can only test a child process'^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^
^^ ^     ^^
 self._shutdown_workers() AssertionError
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^:      ^can only test a child processif w.is_alive():^
 ^
^Exception ignored in: ^  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^  
^ Traceback (most recent call last):
 ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  ^    ^^ ^self._shutdown_workers()^ 

^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
AssertionError^^: ^can only test a child process^
    ^^Exception ignored in: if w.is_alive():^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^
^ ^ ^^Traceback (most recent call last):
 ^^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^ ^^    ^^ ^^self._shutdown_workers() ^
^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    ^    assert self._parent_pid == os.getpid(), 'can only test a child process'^
if w.is_alive():^ ^
 ^^ ^^ ^    ^^^ ^ ^  ^^  ^ ^ 
^^^ ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^^    ^^
^^assert self._parent_pid == os.getpid(), 'can only test a child process'AssertionError: ^
^^can only test a child process ^ ^
^ ^^Exception ignored in:  ^^ ^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
 ^^ Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^         ^self._shutdown_workers()^assert self._parent_pid == os.getpid(), 'can only test a child process'^^

^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^    ^ ^^^if w.is_alive(): ^
^ ^    ^^ ^  ^^   ^^ ^^^ ^ ^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^^AssertionError^^: ^^^can only test a child process^^^^^
^
^^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    Exception ignored in: ^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>

assert self._parent_pid == os.getpid(), 'can only test a child process'^AssertionError
Traceback (most recent call last):
: ^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^ can only test a child process     ^^
self._shutdown_workers() ^^ 
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^Exception ignored in:     ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 
^if w.is_alive(): 
^ Traceback (most recent call last):
 ^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
     ^^ self._shutdown_workers() ^^^ 
^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^^     if w.is_alive():AssertionError^ ^^: 
^ ^^^can only test a child process^  ^^
^^^ ^^ ^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^
^ ^Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^^^    ^^^^^^self._shutdown_workers()
^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^^    if w.is_alive():^^
^assert self._parent_pid == os.getpid(), 'can only test a child process' ^^^
 ^^  ^ 
 
 AssertionError   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
      : ^ assert self._parent_pid == os.getpid(), 'can only test a child process'^
can only test a child process ^  
 ^ ^ ^  ^  ^ Exception ignored in: ^  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^  ^ 
^^
 ^Traceback (most recent call last):
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^self._shutdown_workers()^
^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ ^ ^^     ^if w.is_alive(): 
^^^  ^ ^ ^ ^ ^ ^ ^  ^ ^ ^^ ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^AssertionError
^^: 
can only test a child process^^
AssertionError  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
: ^can only test a child process    ^
assert self._parent_pid == os.getpid(), 'can only test a child process'^
^^ ^ ^  ^ ^  ^ 
 AssertionError :  ^^can only test a child process
^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Exception ignored in: Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
        self._shutdown_workers()
self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
if w.is_alive():    if w.is_alive():
 
           ^ ^ ^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^Exception ignored in: assert self._parent_pid == os.getpid(), 'can only test a child process'<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^

 ^Traceback (most recent call last):
 
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
          Exception ignored in: self._shutdown_workers() assert self._parent_pid == os.getpid(), 'can only test a child process'
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 

   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      if w.is_alive():      
  self._shutdown_workers()  ^
 ^    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^        if w.is_alive():^  ^ ^^
^^^ ^^^ ^ ^^^^ ^^^ ^^^ ^^^^^^ ^^^^^^^^^^^^^^^^^^^^^^^^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^
^^^
 ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^ ^^     ^assert self._parent_pid == os.getpid(), 'can only test a child process' ^
^^ ^ ^  ^ ^
 ^  ^AssertionError  ^ :  ^can only test a child process  ^ ^^ 

^ Exception ignored in: AssertionError : ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^can only test a child process^

^^Traceback (most recent call last):
Exception ignored in: ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^    
^self._shutdown_workers()^
^Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^        ^^if w.is_alive():^self._shutdown_workers()
^^^
  ^^ ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^ ^    if w.is_alive():^^^ ^
^  ^^  ^ ^^ ^^^ ^^^ ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive


^    ^AssertionErrorassert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
: : 
can only test a child process    can only test a child process assert self._parent_pid == os.getpid(), 'can only test a child process'
 
 
Exception ignored in:   Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  
Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
          self._shutdown_workers() 
self._shutdown_workers()  
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
           ^if w.is_alive():^
 if w.is_alive():^  
^^  ^^  ^^  ^^  ^^^  ^ ^^^ ^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^
^    ^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^assert self._parent_pid == os.getpid(), 'can only test a child process'    ^^
^assert self._parent_pid == os.getpid(), 'can only test a child process'^ 
^^ ^ ^^   ^^  ^ ^
  ^AssertionError  : 
 can only test a child processAssertionError   : 
can only test a child process ^
 ^ Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^ Exception ignored in: 
^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^Traceback (most recent call last):
^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
self._shutdown_workers()^^^
    ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^self._shutdown_workers()^
^    ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
if w.is_alive():^
^     ^^if w.is_alive():^  
^ ^ ^  ^ ^^ ^  ^^^ ^ ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^
    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^AssertionError^

:    File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
can only test a child process^     ^
assert self._parent_pid == os.getpid(), 'can only test a child process'  

  AssertionError  : Exception ignored in:   <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>can only test a child process  

 Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  Exception ignored in:     ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 
self._shutdown_workers()^Traceback (most recent call last):
^ 
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^^         ^ ^if w.is_alive():^self._shutdown_workers()^^

^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^^^     ^^^ ^^if w.is_alive(): ^^^ 
^^^  ^  ^^ ^^ ^ ^^ ^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^AssertionError^    : ^assert self._parent_pid == os.getpid(), 'can only test a child process'^

can only test a child process^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive

 ^    ^ Exception ignored in: assert self._parent_pid == os.getpid(), 'can only test a child process'^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 


 Traceback (most recent call last):
 AssertionError   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 :        can only test a child process  self._shutdown_workers()

   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
   Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^     
^^if w.is_alive(): Traceback (most recent call last):

^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^      ^  ^ self._shutdown_workers()  
^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ ^    ^^^^^if w.is_alive():^^
^^ ^^ ^^^^ ^^ ^^^ ^^^ ^^^^ ^^^^^^^
^^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^^^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^^
^^^ ^^ ^^^^^^ ^^^ ^^
 ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^
     ^assert self._parent_pid == os.getpid(), 'can only test a child process' AssertionError^: 
 ^can only test a child process ^ 
  ^ ^^Exception ignored in:  ^^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^ 
^^Traceback (most recent call last):
 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

 ^AssertionError     : ^ ^can only test a child process
 self._shutdown_workers()
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Exception ignored in: ^^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>    ^
^^Traceback (most recent call last):
if w.is_alive():^
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^^     ^^self._shutdown_workers()^ 
^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^     ^ ^ if w.is_alive():^^
^^^^^ ^ ^ ^^^ ^^ ^^^^ ^ ^^^^^^^^^^^^^^^^^^^^^^

^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^AssertionError^^: ^    assert self._parent_pid == os.getpid(), 'can only test a child process'^^can only test a child process^^

 
^ Exception ignored in: ^AssertionError : ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 
can only test a child process
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive

Traceback (most recent call last):
       File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
assert self._parent_pid == os.getpid(), 'can only test a child process'Exception ignored in:  
     <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  self._shutdown_workers() 
 
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^         ^self._shutdown_workers()if w.is_alive(): ^ 

 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^ ^      if w.is_alive():^  ^ 
 ^ ^  ^^  ^^^ ^ ^^ ^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^
^    assert self._parent_pid == os.getpid(), 'can only test a child process'  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^
^^     ^ assert self._parent_pid == os.getpid(), 'can only test a child process'^^ 
^^   ^^^  
^  ^AssertionError ^  : ^ 
 can only test a child process  AssertionError 
:    can only test a child process ^^^^Exception ignored in: 
^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^^Exception ignored in: ^^
^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^Traceback (most recent call last):
^^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    ^^self._shutdown_workers()^    
^self._shutdown_workers()^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^if w.is_alive():^    ^^^if w.is_alive():
^
^^ ^ ^^ ^ ^ ^^   ^^ ^  ^ ^ ^^ ^^^^^^^^^^^^^
^^AssertionError^^: ^^can only test a child process^^^
^^^^
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>AssertionError^Exception ignored in: ^
: ^Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
can only test a child process^
^    Exception ignored in: ^
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>self._shutdown_workers()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    

    self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
assert self._parent_pid == os.getpid(), 'can only test a child process'  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive


    if w.is_alive():    
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 assert self._parent_pid == os.getpid(), 'can only test a child process'  
     if w.is_alive():   
               ^    ^ ^  ^^  ^^ ^^  ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^^    ^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^^^
^^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^ ^^     ^^ assert self._parent_pid == os.getpid(), 'can only test a child process'^^
 ^ ^^    ^^ ^^  ^^^  ^^  ^^  ^^^ ^^
^^ AssertionError^^: ^ ^^can only test a child process^^^
^^^
^^Exception ignored in: AssertionError^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^: 
^can only test a child process^^Traceback (most recent call last):

^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^Exception ignored in: ^^self._shutdown_workers()<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>

^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    if w.is_alive():^
^self._shutdown_workers()^ ^^
 ^ ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^ ^if w.is_alive():^^ 
^ ^ ^^  ^^^ ^^^^^ ^^^ ^^^ ^^^^ ^^^^^^
^^^AssertionError^^
: AssertionError^^can only test a child process^^
: can only test a child process^
^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^

   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
      assert self._parent_pid == os.getpid(), 'can only test a child process' 
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

AssertionErrorAssertionError: : can only test a child processcan only test a child process

Epoch  4/15  train=0.0021  val=0.0020  lr=4.45e-04  
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
        self._shutdown_workers()self._shutdown_workers()

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
        if w.is_alive():if w.is_alive():

            ^^^ ^ ^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
assert self._parent_pid == os.getpid(), 'can only test a child process'
     assert self._parent_pid == os.getpid(), 'can only test a child process'  
                 ^ ^^ ^^^^^^^^^^^^^^^^^^^^^^^^Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^^Exception ignored in: Traceback (most recent call last):
^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^
^    ^^Traceback (most recent call last):
^^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    ^    ^self._shutdown_workers()if w.is_alive():^^^

^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ ^^    ^ ^^ if w.is_alive():^

 AssertionError ^ : ^ ^ can only test a child process ^ 
^^ Exception ignored in: ^^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^

 AssertionError^Traceback (most recent call last):
:  ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^can only test a child process^^    
^^self._shutdown_workers()
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Exception ignored in: ^    <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^if w.is_alive():^
Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^^ ^     ^self._shutdown_workers()^ 

   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^       File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

 ^assert self._parent_pid == os.getpid(), 'can only test a child process'    
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
if w.is_alive(): ^    
 ^ assert self._parent_pid == os.getpid(), 'can only test a child process'^
  ^    ^   ^ ^   ^ ^   ^ ^ ^ 
 ^^ ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^^ assert self._parent_pid == os.getpid(), 'can only test a child process'^
^ ^ ^  ^^^  ^^^^ ^^^^ ^^^^ ^^^
^ ^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
     ^^assert self._parent_pid == os.getpid(), 'can only test a child process' ^^ 
^^  ^^^ ^^^ ^^^^^ ^^^^^ ^ ^^^ ^^^^ ^^ ^^^^ ^^ ^^^^^^^^^^^^^^^^
^^^AssertionError^^: ^^^can only test a child process^^
^^^^^^Exception ignored in: ^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
^Traceback (most recent call last):
^
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^self._shutdown_workers()AssertionError^^
^:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^can only test a child process    ^^
^if w.is_alive():^
^^^^Exception ignored in: ^^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^ ^

AssertionError ^Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
: ^^can only test a child process     ^^
 ^self._shutdown_workers()
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^Exception ignored in:     ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>if w.is_alive():^

Traceback (most recent call last):
AssertionError
^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
: ^     can only test a child process^
 self._shutdown_workers() ^
^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^    
^^if w.is_alive():^^Traceback (most recent call last):


^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^     self._shutdown_workers()assert self._parent_pid == os.getpid(), 'can only test a child process'^
 
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^   ^      ^^if w.is_alive():^ ^
 ^^ 
  ^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
   ^         assert self._parent_pid == os.getpid(), 'can only test a child process' ^
  ^^^ ^^^^^ ^^ ^ ^^ ^^^^^ 
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^ ^assert self._parent_pid == os.getpid(), 'can only test a child process'^^ 
^^  ^^  ^ ^ 
 ^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^ ^^ assert self._parent_pid == os.getpid(), 'can only test a child process'^^
 ^^   ^^  ^^^^ ^ ^^ ^^^^^ ^^ ^^^^^^^ ^ ^^^^^^ ^^^^ ^^^^^^^^^^^^^^
^^^AssertionError^^: ^^can only test a child process^^^
^^^^^^^^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^

Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
AssertionError^^:     ^self._shutdown_workers()^^^can only test a child process^^
^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    ^^Exception ignored in: if w.is_alive():^^
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^ ^
^^  Traceback (most recent call last):
^^  
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
AssertionError     ^self._shutdown_workers(): 
can only test a child process ^
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^Exception ignored in: ^^    ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>if w.is_alive():^^

Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^ ^     
^self._shutdown_workers() ^AssertionError
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^:  ^    can only test a child processif w.is_alive():^ 


   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  Exception ignored in:      ^assert self._parent_pid == os.getpid(), 'can only test a child process' <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^ 

^ Traceback (most recent call last):
 ^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      ^ ^^self._shutdown_workers() ^^ 
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ ^^     ^if w.is_alive():^ 
^ ^ 
 ^ ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
     ^^ assert self._parent_pid == os.getpid(), 'can only test a child process'^
 ^^ ^ ^   
^  ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^^     ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^ ^^
 ^^  ^^ ^^   ^ ^^^ ^^ ^^^ ^^^
 ^^^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^ ^    ^ ^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^^^
^^^ ^^^ ^ ^^^ ^^^ ^^^ ^^^ ^^^^ ^^ ^^^ ^^^^^ ^^^^^
^^^^^^AssertionError^^^^: ^^^^can only test a child process^^^
^^^^^^Exception ignored in: ^
^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>AssertionError
^: ^^Traceback (most recent call last):
can only test a child process^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^Exception ignored in:     ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^self._shutdown_workers()
^Traceback (most recent call last):
^^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^        ^self._shutdown_workers()
^if w.is_alive():AssertionError^
^:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^
^    can only test a child process ^ if w.is_alive():
^
 ^  ^ Exception ignored in:   <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^ 
Traceback (most recent call last):
^    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^ ^    
 self._shutdown_workers()^AssertionError^
^: ^^can only test a child process  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^
^    ^^if w.is_alive():^^^Exception ignored in: 
^^^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^ 
 
^Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    ^ assert self._parent_pid == os.getpid(), 'can only test a child process'     
 
self._shutdown_workers()  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^ 
     ^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^assert self._parent_pid == os.getpid(), 'can only test a child process'     ^
^if w.is_alive():  ^^ 
  ^  ^    ^  ^      ^ ^^  
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^      assert self._parent_pid == os.getpid(), 'can only test a child process'^^
 ^ ^^^ ^^ ^^^^^ ^^ ^^^ ^^^ ^^ ^^^^^ 
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^^     ^^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^
^^^ ^^^^^^ ^ ^^^^^ ^^^^ ^^^ ^^^ ^^^  ^^^ ^^
AssertionError ^^^^: ^^can only test a child process^^^^
^^^^^^^^
^^^^Exception ignored in: ^AssertionError^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^: 
^can only test a child process^^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^self._shutdown_workers()^
^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

AssertionErrorif w.is_alive():^^:     
^can only test a child processself._shutdown_workers()
^ 
 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Exception ignored in:      ^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>if w.is_alive():
^
  ^Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^ ^      ^^self._shutdown_workers()^ 
 ^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^
^^    AssertionError^if w.is_alive():^: ^^
^^ can only test a child process ^^
^^ ^^ ^Exception ignored in: ^ ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^^ 

^Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^^    
assert self._parent_pid == os.getpid(), 'can only test a child process'self._shutdown_workers()^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^     assert self._parent_pid == os.getpid(), 'can only test a child process'    ^ if w.is_alive():
 ^  ^
 ^   ^   ^  
    File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
       assert self._parent_pid == os.getpid(), 'can only test a child process'    
   ^  ^  ^^  ^^ ^^^  ^^^ ^^ ^ ^^^ ^^^^^^^^^^^^^^^^^^^^^^
^^^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^^^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^^
^^^^ ^^^ ^^^^^^ ^^ ^^^ ^^^^ ^^^^ ^^^^^ ^^ ^^^^^ ^^^^ ^^^
^^AssertionError^
^^AssertionError: ^^: can only test a child process^^can only test a child process
^^

AssertionError^Exception ignored in: Exception ignored in: : <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00><function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
can only test a child process
^
Traceback (most recent call last):
^Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Exception ignored in:     ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>self._shutdown_workers()^    ^

^Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
if w.is_alive():^
^    
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ self._shutdown_workers()     
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^    if w.is_alive(): 
if w.is_alive():^   
^  ^   ^    ^ ^^  ^^^^^ ^^^^ ^^^^^^^^^^^^^^
^^^AssertionError^^^: ^^^^can only test a child process^^^


^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
      File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^assert self._parent_pid == os.getpid(), 'can only test a child process'    ^
assert self._parent_pid == os.getpid(), 'can only test a child process' 
 ^Exception ignored in:  
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
Traceback (most recent call last):
        File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      assert self._parent_pid == os.getpid(), 'can only test a child process' 
self._shutdown_workers()  
           File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
       ^  if w.is_alive():^ 
 ^  ^^ ^ ^^ ^^ ^^ ^ ^  ^ ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^    ^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^
^^ ^^^ ^^  ^^^ ^^^^^ ^^^^^  ^
^AssertionError^ ^
:  AssertionError^can only test a child process: ^ 
can only test a child process^
^^^^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^Exception ignored in: 
^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>

^Traceback (most recent call last):
AssertionErrorTraceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
: ^^can only test a child process    ^^    
self._shutdown_workers()^self._shutdown_workers()
^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^if w.is_alive():
if w.is_alive():^ 
^ ^   ^ ^    ^    ^^^^^^^^^^^^^^^^^^^^^^
^^AssertionError^^: ^^can only test a child process^^^
^

  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
        assert self._parent_pid == os.getpid(), 'can only test a child process'assert self._parent_pid == os.getpid(), 'can only test a child process'

                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^AssertionError^^: ^can only test a child process
^^
AssertionError: can only test a child process
Epoch  5/15  train=0.0021  val=0.0020  lr=4.06e-04  *
Epoch  6/15  train=0.0021  val=0.0020  lr=3.58e-04  *
Epoch  7/15  train=0.0021  val=0.0020  lr=3.06e-04  *
Epoch  8/15  train=0.0021  val=0.0020  lr=2.50e-04  
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    Exception ignored in: assert self._parent_pid == os.getpid(), 'can only test a child process'<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>

 Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
     self._shutdown_workers() 
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
        if w.is_alive(): 
    ^   ^ ^^^^^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^^
 ^ ^ ^ ^ ^^  ^ ^^ ^ ^ ^^^^^^^
^AssertionError: ^^can only test a child process^
^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^^self._shutdown_workers()^
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^if w.is_alive():^
^ ^  ^^ ^^^ ^  
AssertionError^^^: ^can only test a child process^
^^Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    
self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    if w.is_alive():    
assert self._parent_pid == os.getpid(), 'can only test a child process'  
          ^ ^   ^^ ^^^ ^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^
^^^ ^ ^ ^ ^^  ^ ^^^  ^ ^ ^^^^^^^^^^^^^
^AssertionError^^: ^can only test a child process
^^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    
^self._shutdown_workers()^^^if w.is_alive():      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^
 ^^  ^^ ^^ 
 AssertionError : can only test a child process^
^^^Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    ^^self._shutdown_workers()
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^
      File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    if w.is_alive():
assert self._parent_pid == os.getpid(), 'can only test a child process'
            ^^ ^ ^ ^ ^ ^ ^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^^ ^ ^  ^ ^  ^  ^ ^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^AssertionError: ^can only test a child process^
^^^^Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^self._shutdown_workers()
AssertionError
:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
can only test a child process    
if w.is_alive():
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  
Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
       self._shutdown_workers()^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    if w.is_alive():^^
^ ^^ ^ ^ ^ ^ ^
 ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'
^^  ^^  ^^   ^ ^
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
      ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^  ^ ^^ ^ ^^ ^^ ^ ^ ^ ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^
^AssertionError: ^^can only test a child process
^^^^^Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
^Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^if w.is_alive():^
^  ^ ^
   AssertionError : ^can only test a child process^^
^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
^Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

      File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    if w.is_alive():assert self._parent_pid == os.getpid(), 'can only test a child process'

           ^ ^^ ^ ^^ ^  ^ ^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    assert self._parent_pid == os.getpid(), 'can only test a child process'^
^^ ^^ ^ ^ ^  ^^ ^  ^ ^ ^^^^^^^^^^^^^^^^^
^^AssertionError^: ^can only test a child process^^^
^^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^    if w.is_alive():^^
 
 AssertionError :  can only test a child process
  Exception ignored in:  ^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
^Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^self._shutdown_workers()^^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    ^if w.is_alive():

  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
     assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^ ^   ^ ^^ ^^^^^^^^^^^^^
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^assert self._parent_pid == os.getpid(), 'can only test a child process'
^  ^ ^  ^ ^ ^^ ^  ^ ^^^^^^^^^^^^^^^^^^^^^^
^AssertionError^: ^^can only test a child process
^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^^^self._shutdown_workers()^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^    
AssertionError: can only test a child process
if w.is_alive():
  Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  
 Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^^self._shutdown_workers()^
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    ^if w.is_alive():^^

    File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
      assert self._parent_pid == os.getpid(), 'can only test a child process'  
  ^ ^^ ^ ^ ^ ^ ^ ^ ^ ^ ^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^^
 ^ ^  ^ ^  ^ ^ ^  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError^^: ^^can only test a child process^
^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Exception ignored in: Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

    self._shutdown_workers()Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
        if w.is_alive():self._shutdown_workers()Exception ignored in: 

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 
 Traceback (most recent call last):
       File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
if w.is_alive(): 
        self._shutdown_workers()
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^    
 Traceback (most recent call last):
if w.is_alive():^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

      ^^ ^self._shutdown_workers()^ ^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  ^^^     ^^^if w.is_alive():^ ^
^^^^^ ^^^^^ ^^
^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  ^^          ^assert self._parent_pid == os.getpid(), 'can only test a child process' ^
assert self._parent_pid == os.getpid(), 'can only test a child process'^ ^
^  ^^^  ^
    File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^^       assert self._parent_pid == os.getpid(), 'can only test a child process'  ^
 ^   ^    ^   ^^  ^^  ^ 
^^  ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^ ^     ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^^
^^^ ^^^^^^ ^^^^^^^ ^ ^^^^ ^^^ ^^^^^ ^^ ^^ ^^ ^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^AssertionError^^
: 
^AssertionError: ^can only test a child processAssertionError
^: can only test a child process^
Exception ignored in: can only test a child process^
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^Exception ignored in: ^
Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00><function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>Traceback (most recent call last):
^

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^Traceback (most recent call last):
Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    ^    self._shutdown_workers()

self._shutdown_workers()    AssertionError:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

self._shutdown_workers()can only test a child process  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
        

if w.is_alive():
Exception ignored in: if w.is_alive():  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>     Traceback (most recent call last):
 
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 if w.is_alive():      
 self._shutdown_workers() 
     File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^      ^ ^ if w.is_alive(): ^ 
^^   ^  ^^^  ^ ^^^ ^^
^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^     ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^^^^^^^^ ^^^^^ ^^^ ^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^^     ^assert self._parent_pid == os.getpid(), 'can only test a child process'^

 ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  ^      assert self._parent_pid == os.getpid(), 'can only test a child process'   ^ ^
    ^
  ^^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^   ^     ^ ^assert self._parent_pid == os.getpid(), 'can only test a child process' ^^ ^^
 ^^ ^  ^^  ^^^ ^^ ^^^^ ^^ ^ ^^^^^ ^^^^ ^^ ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^AssertionError^^^: ^^^^can only test a child process^^

^^AssertionError^^^^: ^^can only test a child process^Exception ignored in: ^
^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^^
^Exception ignored in: ^Traceback (most recent call last):

<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^
AssertionError^    ^Traceback (most recent call last):
:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
can only test a child processself._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

    self._shutdown_workers()

AssertionError    Exception ignored in: if w.is_alive()::   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
    can only test a child process if w.is_alive():


 Traceback (most recent call last):
     File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  
     Traceback (most recent call last):
 self._shutdown_workers()    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^         self._shutdown_workers()^^if w.is_alive():

^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^     ^ if w.is_alive():^^
^ ^  ^^ ^ ^ ^^ ^  ^^ ^ ^^^ ^^
^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^        ^^assert self._parent_pid == os.getpid(), 'can only test a child process'assert self._parent_pid == os.getpid(), 'can only test a child process'^^

^^ ^ ^ ^ ^^  ^^^  ^^  

    File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
          assert self._parent_pid == os.getpid(), 'can only test a child process'  assert self._parent_pid == os.getpid(), 'can only test a child process' 
 
      ^  ^^  ^ ^ ^^^   ^ ^ ^^ ^ ^ ^  ^^^  ^^ ^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^AssertionError^^^: ^^^can only test a child process^^
^^
AssertionError^^: Exception ignored in: can only test a child process^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
^^
^^Traceback (most recent call last):
Exception ignored in: ^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^
^^^Traceback (most recent call last):
    ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^
^self._shutdown_workers()AssertionError
^    self._shutdown_workers()
:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

can only test a child process  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    AssertionError
: if w.is_alive():    can only test a child processif w.is_alive():

Exception ignored in: 
 Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00><function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>   
  
Traceback (most recent call last):
  Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
          ^self._shutdown_workers()self._shutdown_workers() ^

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^^if w.is_alive():^    
^if w.is_alive():^ 
^^  ^^  ^  ^  ^ ^  ^ ^ ^^^^^
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^

^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^     ^^^assert self._parent_pid == os.getpid(), 'can only test a child process' ^
 ^ ^ ^^  ^^^  
^  ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
     ^  
 assert self._parent_pid == os.getpid(), 'can only test a child process'    File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  ^ 
    ^  ^assert self._parent_pid == os.getpid(), 'can only test a child process'^  
^ ^   ^^  ^ ^ ^^^  ^ ^^^^   ^  ^^ ^^ ^ ^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^^AssertionError^^^^^: ^^can only test a child process^^^^
^^^^^^Exception ignored in: ^
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^AssertionError^
^: ^^^Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^can only test a child process
^    ^
^self._shutdown_workers()AssertionErrorException ignored in: ^: ^
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
can only test a child process
Traceback (most recent call last):

if w.is_alive():      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
AssertionError: 
 Exception ignored in:     can only test a child process
 <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> self._shutdown_workers()
 
 Traceback (most recent call last):
Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

     Traceback (most recent call last):
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^if w.is_alive():    ^self._shutdown_workers()
self._shutdown_workers()
 
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
     ^if w.is_alive():      ^
if w.is_alive():  
^    ^ ^  ^^^  ^ ^ ^^ ^^ ^ ^^^^^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^^
^ ^^^ ^
^^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^ ^^    assert self._parent_pid == os.getpid(), 'can only test a child process'^ ^^
 
^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  ^      ^assert self._parent_pid == os.getpid(), 'can only test a child process'  
 
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
         ^  assert self._parent_pid == os.getpid(), 'can only test a child process'^  ^
  ^  ^ ^^  ^^^   ^^^  ^^^ ^ ^^ ^^ ^^ ^^^^^ ^^ ^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^
^AssertionError^AssertionError: ^^can only test a child process: ^^can only test a child process
^^^
^^Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^^Traceback (most recent call last):
^
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Traceback (most recent call last):

^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^AssertionError        ^: self._shutdown_workers()self._shutdown_workers()can only test a child process^



  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
AssertionError        : Exception ignored in: if w.is_alive():
can only test a child process<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>if w.is_alive(): 


Exception ignored in:  Traceback (most recent call last):
  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

      Traceback (most recent call last):
  self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^     ^     self._shutdown_workers()if w.is_alive():^ 

^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ ^     if w.is_alive(): ^ ^^
 ^^ ^ ^ ^^ ^ ^^ ^^^
 ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^ ^     ^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^^
^
^ ^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^    ^ assert self._parent_pid == os.getpid(), 'can only test a child process'^ ^^ ^
^^  ^^
 ^    File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
     ^  ^ assert self._parent_pid == os.getpid(), 'can only test a child process'  
 
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  ^    assert self._parent_pid == os.getpid(), 'can only test a child process'  ^  
^   ^^  ^^ ^  ^ ^^   ^^^  ^ ^^ ^^ ^^ ^^^^^^ ^^^^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^^AssertionError
^: ^can only test a child process^^AssertionError
^^: ^^^^can only test a child process^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
^^^
Traceback (most recent call last):

^Exception ignored in: AssertionError  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^: self._shutdown_workers()
can only test a child process^^Traceback (most recent call last):



  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
AssertionError        : can only test a child processif w.is_alive():Exception ignored in: 
self._shutdown_workers()<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>


   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Traceback (most recent call last):
        File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
if w.is_alive(): 
Exception ignored in:       self._shutdown_workers()<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  
  
^Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
     ^     if w.is_alive():self._shutdown_workers()^^

^ ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^     ^^if w.is_alive(): ^
^^  ^  ^^  ^ ^ ^^ ^^^ 
^^ ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^^    
^^assert self._parent_pid == os.getpid(), 'can only test a child process'  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^
^     ^assert self._parent_pid == os.getpid(), 'can only test a child process' ^
^^ ^ ^ ^^ 
^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^^       ^ assert self._parent_pid == os.getpid(), 'can only test a child process' 
  
     File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
          assert self._parent_pid == os.getpid(), 'can only test a child process'  ^ 
^  ^ ^  ^  ^ ^   ^^ ^ ^ ^ ^^^^ ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^
AssertionError^AssertionError^: ^: ^can only test a child process^can only test a child process^^
^^
^^Exception ignored in: ^
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^AssertionError^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
^: ^Traceback (most recent call last):


AssertionErrorcan only test a child process  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
: 
can only test a child processTraceback (most recent call last):
    
self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

Exception ignored in:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>    <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>    
if w.is_alive():Traceback (most recent call last):


  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Traceback (most recent call last):
 self._shutdown_workers()      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 self._shutdown_workers()
     
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
self._shutdown_workers()     
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
if w.is_alive(): 
      if w.is_alive():      
if w.is_alive():     
  ^   ^  ^^  ^^  ^^^^ ^^^^^ ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^
^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^            assert self._parent_pid == os.getpid(), 'can only test a child process'assert self._parent_pid == os.getpid(), 'can only test a child process'^assert self._parent_pid == os.getpid(), 'can only test a child process'



   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
         assert self._parent_pid == os.getpid(), 'can only test a child process'    
                           ^    ^^ ^^ ^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^AssertionError
^: ^^AssertionErrorcan only test a child process^
: ^
AssertionErrorcan only test a child process^
: ^can only test a child process

AssertionError: can only test a child process
Epoch  9/15  train=0.0021  val=0.0020  lr=1.94e-04  *
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Traceback (most recent call last):
    self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    self._shutdown_workers()if w.is_alive():

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
        if w.is_alive(): 
   ^ ^ ^^ ^Exception ignored in:  ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
 ^Exception ignored in: Traceback (most recent call last):
^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^^    
^^self._shutdown_workers()^Traceback (most recent call last):


  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
assert self._parent_pid == os.getpid(), 'can only test a child process'    ^^self._shutdown_workers()    
^
if w.is_alive(): ^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^   ^       ^ if w.is_alive():
     File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive

        assert self._parent_pid == os.getpid(), 'can only test a child process'  ^
   ^  ^^ ^^   ^ ^   ^^^^ ^ ^^^^ ^ ^^^ ^^ ^^^^^^^^^^
^^^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^

  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^     ^assert self._parent_pid == os.getpid(), 'can only test a child process' ^
^ ^ ^  ^^ ^ ^^ ^ ^ ^  ^^ ^^   ^^  ^ ^^ ^^^^^^ ^
^^^^^AssertionError: ^^^^^can only test a child process^^^
^^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
^Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    ^^^self._shutdown_workers()^^^
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^    
^^AssertionError^if w.is_alive():: ^^
^can only test a child process^^
 ^^^^ Exception ignored in:  ^^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^ 
^^^ Traceback (most recent call last):
^ ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^^    ^^^self._shutdown_workers()^
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^^^    ^if w.is_alive():^


AssertionError^  AssertionError^:  : can only test a child process ^can only test a child process^
 

 Exception ignored in:  Exception ignored in:   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>    
^
Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Traceback (most recent call last):
assert self._parent_pid == os.getpid(), 'can only test a child process'^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
         ^self._shutdown_workers()self._shutdown_workers() ^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 
    ^ if w.is_alive():  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    
 if w.is_alive():^
  ^   ^  
    File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
           assert self._parent_pid == os.getpid(), 'can only test a child process'  
^   ^^^ ^^^ ^^^ ^^^^ ^^^^ ^^^^ ^^ ^^^ ^^^ ^^ ^^^
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^

^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^^     ^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^ ^
 ^ ^^ ^ ^ ^ ^ ^  ^^^ ^ ^ ^^ ^  ^^^^ ^^ ^^ 
^^ AssertionError: ^^^can only test a child process^
^^^^^^Exception ignored in: ^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
^^^^^Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    ^^^self._shutdown_workers()^^

^^AssertionError^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
: ^^    ^^can only test a child process^^if w.is_alive():

^^^ ^ ^ ^^^ Exception ignored in: ^^ ^ ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
^^ ^^^Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^^    ^^^self._shutdown_workers()^^
^^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    ^^^^^
^AssertionError^if w.is_alive():^^: 

 can only test a child processAssertionError
 : 
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 can only test a child process     
assert self._parent_pid == os.getpid(), 'can only test a child process'
 Exception ignored in:   <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> Exception ignored in: ^
 <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^Traceback (most recent call last):

 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^          ^ ^ self._shutdown_workers()self._shutdown_workers()^ 

^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^    ^    
if w.is_alive():if w.is_alive():^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive

^
     ^  assert self._parent_pid == os.getpid(), 'can only test a child process'^  
 ^ ^     ^   ^ ^  ^^ ^ ^^^^ ^ ^^ ^^^^^ ^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^    ^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^
assert self._parent_pid == os.getpid(), 'can only test a child process'^
 
^AssertionError    ^:  ^ can only test a child process
^   ^ Exception ignored in:  ^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  ^
 ^Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  ^^      ^self._shutdown_workers() ^^^^
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    ^^^^if w.is_alive():^^
^^^ ^^ ^^^^^  ^^^^ ^^
^^ AssertionError^^ ^^^: ^^^^^can only test a child process^^^
^^^^^^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^^^^^
^^^^Traceback (most recent call last):
^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^        ^^assert self._parent_pid == os.getpid(), 'can only test a child process'self._shutdown_workers()^^

^ ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ ^^^     ^if w.is_alive():^ ^
 
^ AssertionError^  :  
   can only test a child process AssertionError
:    can only test a child process  Exception ignored in: 
^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>Exception ignored in: ^^
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^^Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    ^^self._shutdown_workers()    ^
^self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^
^      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^if w.is_alive():^    
^^if w.is_alive():^ ^
  ^^ 
    File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^  ^      ^assert self._parent_pid == os.getpid(), 'can only test a child process'   
^ ^  ^^^^^^ ^ ^^^ ^^ ^^^^ ^^^^  ^^^ ^^^^^ ^^^^^^^
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^
assert self._parent_pid == os.getpid(), 'can only test a child process'^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive

^    
AssertionErrorassert self._parent_pid == os.getpid(), 'can only test a child process' ^ : 
^   can only test a child process ^ ^
 ^   ^Exception ignored in:  ^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^  
^ ^Traceback (most recent call last):
^  ^    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^      ^^^^self._shutdown_workers()^^^^^
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^^^    ^^^^if w.is_alive():^^
^^^^^ ^^^ ^^ 
^^AssertionError^ ^:  ^^ can only test a child process^^^ 
^^^^^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^^
^Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    ^^self._shutdown_workers()^^
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^    ^^^if w.is_alive():^^
^^^ ^ ^^ ^
 
^AssertionError   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^ ^:      
can only test a child processassert self._parent_pid == os.getpid(), 'can only test a child process'
^AssertionError
: ^Exception ignored in: can only test a child process ^ ^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
^ 
 ^Traceback (most recent call last):
^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^ ^ ^     self._shutdown_workers()Exception ignored in: ^
 <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

^
      File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^Traceback (most recent call last):
    ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
if w.is_alive():assert self._parent_pid == os.getpid(), 'can only test a child process'^
    
^self._shutdown_workers()  
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  ^      ^if w.is_alive(): ^
  ^  ^     ^  ^ ^ ^   ^^^ ^^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^     ^^assert self._parent_pid == os.getpid(), 'can only test a child process' ^ ^
  ^ ^  ^^  ^^ ^  ^
  ^AssertionError^ : ^^ can only test a child process^ ^
^^ ^^ ^Exception ignored in:  ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^ ^^^^
Traceback (most recent call last):
^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^^^    ^self._shutdown_workers()^^
^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^^    ^^^^if w.is_alive():^^^
^
 ^^AssertionError^:  ^^ can only test a child process^
^ ^ Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^ ^
 ^^Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    ^self._shutdown_workers()^^
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^
^AssertionError^    : ^^if w.is_alive():can only test a child process^^

^^ ^^Exception ignored in:  ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^^ 
^^^ Traceback (most recent call last):
 ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 
    
 ^AssertionError  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
self._shutdown_workers()    
: ^can only test a child process  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
assert self._parent_pid == os.getpid(), 'can only test a child process'^

    ^if w.is_alive(): Exception ignored in: ^
 <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^   
^ ^ Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^   ^      ^^self._shutdown_workers() ^^
 
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^if w.is_alive():    ^^^
 ^^assert self._parent_pid == os.getpid(), 'can only test a child process' ^^
  ^^^  ^  ^  ^^  ^^ ^^^ ^
^ ^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^    ^ assert self._parent_pid == os.getpid(), 'can only test a child process'^^
 ^^ ^ ^^ ^^^  ^^^ ^^^ ^^^ ^^ ^^
^ ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^     ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^
^^^^ ^^ ^^^ ^^^ ^^^^ 
^^AssertionError^^:  ^ ^^can only test a child process ^^
^ ^^^ ^Exception ignored in:  ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
^^Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^^    ^^^self._shutdown_workers()^^
^^^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^
^    ^AssertionErrorif w.is_alive():: ^
^^^can only test a child process
 ^ ^ ^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^^ 
^ Traceback (most recent call last):
^^ ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^
^    ^AssertionError^self._shutdown_workers()^
^: ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
can only test a child process^^    ^
^if w.is_alive():^
^^ Exception ignored in: ^^  ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^ 
^^Traceback (most recent call last):
 ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 
^       File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    self._shutdown_workers()
^
assert self._parent_pid == os.getpid(), 'can only test a child process'AssertionError^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

: ^     ^if w.is_alive():can only test a child process ^

 ^    ^ ^Exception ignored in:   ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  ^
  ^ Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 
^       File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^self._shutdown_workers()^    
^assert self._parent_pid == os.getpid(), 'can only test a child process'^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^
^ ^     ^^ if w.is_alive():^^ ^^
^^  ^  ^ ^   ^^ ^
   ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  ^^    ^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^
^^^^ ^^ ^^^^^ ^^^ ^^ ^^^^^ ^^ ^^^ ^^^^  ^^^^
 ^^^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^^^    
^AssertionError^^assert self._parent_pid == os.getpid(), 'can only test a child process': ^
^can only test a child process^^
 ^^ Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^
 ^^ ^^Traceback (most recent call last):
 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^      ^^^self._shutdown_workers()^
^ ^^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^     ^
^if w.is_alive():^AssertionError^
: ^ ^can only test a child process ^^
 ^^^^  ^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^^ 
^^^^^Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^
    ^^AssertionErrorself._shutdown_workers()^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^: ^^    can only test a child process^^if w.is_alive():
^^^
^ ^^Exception ignored in:  
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^ 
Traceback (most recent call last):
 ^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
         self._shutdown_workers() assert self._parent_pid == os.getpid(), 'can only test a child process'^
 
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^     ^^ ^if w.is_alive():^ ^^^^ 
  ^^  ^ ^^ ^^ ^ 
  ^ AssertionError  ^:  ^
can only test a child process^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^^    Exception ignored in: ^^assert self._parent_pid == os.getpid(), 'can only test a child process'<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^

^^ ^Traceback (most recent call last):
^^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^     ^^self._shutdown_workers() ^
^ ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^     ^ 
if w.is_alive():^ ^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive

 ^      ^^ ^^assert self._parent_pid == os.getpid(), 'can only test a child process' ^^
 ^^  ^^^ ^^^ ^^^^ ^^^^ ^ ^^^^^ ^^^^ ^^^ 
^^ AssertionError^ ^^:  can only test a child process^^^
^
^^^Exception ignored in: ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>    ^^^^^
assert self._parent_pid == os.getpid(), 'can only test a child process'^Traceback (most recent call last):
^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^^ ^^     ^^ ^self._shutdown_workers()^ 
^^ ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^^    
 ^if w.is_alive():AssertionError^ 
^ :  ^can only test a child process^  ^
 ^^^ ^Exception ignored in:  ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^ ^
 ^^Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^^^^    ^^^self._shutdown_workers()
^^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^    ^^if w.is_alive():^
^
^AssertionError ^: ^ 
^can only test a child process   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^
 ^     assert self._parent_pid == os.getpid(), 'can only test a child process'^ 
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^  
^ ^Traceback (most recent call last):
^ ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^     ^^^ self._shutdown_workers()^^ 
 ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^     ^^ ^if w.is_alive():^
^ ^ 
^  AssertionError^^^:  ^can only test a child process^^ 

^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
     ^Exception ignored in: ^ ^assert self._parent_pid == os.getpid(), 'can only test a child process'<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 
^^ 
^^Traceback (most recent call last):
 ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^^     ^^ ^self._shutdown_workers()^^ 
^^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^     ^^ ^^if w.is_alive(): ^^ 
^ 
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  ^^^^     ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^ 
^^  ^ ^^ ^^^ ^^^ ^^^ 
^AssertionError^ ^ ^: ^ can only test a child process^^
^ ^^ ^^ ^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^    ^^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^
^ ^ ^^ ^^ ^^ ^^^ ^^ 
^AssertionError ^:  can only test a child process^
 ^ ^^^^^^^^^^^^^^^^^^
^AssertionError^: ^can only test a child process^
^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Exception ignored in:     <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>if w.is_alive():

Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
     self._shutdown_workers() 
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
     if w.is_alive():
  ^ ^ Exception ignored in:  ^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^ 
^ Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    ^self._shutdown_workers()^^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    ^^if w.is_alive():^^
^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
 ^  
    File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
         assert self._parent_pid == os.getpid(), 'can only test a child process'^ 
^ ^   ^   ^     ^^^^ ^^ ^^Exception ignored in:  ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 
^^ Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^self._shutdown_workers()^
^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^if w.is_alive():^
^^
 ^^ ^ ^ ^ ^^  ^^ ^  ^^^ ^ ^  ^ ^ ^ ^^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^
^AssertionError    ^assert self._parent_pid == os.getpid(), 'can only test a child process': ^^^^can only test a child process

^ ^
 AssertionError^Exception ignored in: :  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^
can only test a child process^ Traceback (most recent call last):
^
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^     Exception ignored in: ^ self._shutdown_workers()<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^

 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Traceback (most recent call last):
^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    ^^    ^if w.is_alive():^^
self._shutdown_workers()^^ ^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ ^     ^^if w.is_alive(): ^^
 ^
 ^ AssertionError^  : ^ ^can only test a child process ^
 ^^^^Exception ignored in:  ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^^ Traceback (most recent call last):
^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    ^self._shutdown_workers()^^^^
^^^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

^      File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    if w.is_alive():assert self._parent_pid == os.getpid(), 'can only test a child process'^^
^
  ^ ^^   ^ ^ ^ ^  ^
  ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^ ^     
^ assert self._parent_pid == os.getpid(), 'can only test a child process'^AssertionError 
  ^: ^ can only test a child process^
 ^^ ^Exception ignored in: ^  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^ 
^Traceback (most recent call last):
^  ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^^     self._shutdown_workers()
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive

^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
        assert self._parent_pid == os.getpid(), 'can only test a child process'^^if w.is_alive():^
^^ ^^^ 
^ ^ ^^  ^^^  ^ ^  ^^ ^ ^ ^^^^ ^ ^^^ ^^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

AssertionError^: ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^can only test a child process^^
    ^^^
assert self._parent_pid == os.getpid(), 'can only test a child process'
^^ Exception ignored in: AssertionError^^: ^ can only test a child process <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^

 ^Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 Exception ignored in: ^      <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^self._shutdown_workers()
^^
 Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    ^^if w.is_alive():^^    ^
 ^self._shutdown_workers()^ 
^
AssertionError   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^     ^if w.is_alive()::   ^
 can only test a child process ^
 ^^^ ^^ ^ ^^Exception ignored in:  ^ ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
^^^^Traceback (most recent call last):
^^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^^^^self._shutdown_workers()^^^

^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^if w.is_alive():    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^ ^

^    File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^      ^ ^assert self._parent_pid == os.getpid(), 'can only test a child process'^  

 AssertionError   :     can only test a child process 
^   ^Exception ignored in:  ^  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^
  ^ Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^ ^     self._shutdown_workers()^
^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    ^^if w.is_alive():^^^

^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^^     ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^ 
 ^^  ^^ ^ ^ ^ ^^^^^ ^^ ^^^ ^^^^^^ ^^^^ ^^ ^^^^ ^^ ^^^^^^^^^^^^
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^    ^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^^^
^
 ^AssertionError^ : ^^ ^can only test a child process^^ ^
^ 
Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^AssertionError
^ : Traceback (most recent call last):
^ can only test a child process  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^ 
    ^ self._shutdown_workers()^^
Exception ignored in:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>    ^
^if w.is_alive():Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

^ ^ ^^    ^^ ^^self._shutdown_workers()^^
 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ ^^ ^     ^if w.is_alive():^^

^AssertionError ^^ ^:  ^^ can only test a child process^^ 
^^ ^Exception ignored in:  ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^^
^^^^^Traceback (most recent call last):
^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^^^self._shutdown_workers()
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^
    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

    ^^ if w.is_alive():^^
^ 
 
 AssertionError   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
      : can only test a child processassert self._parent_pid == os.getpid(), 'can only test a child process'  

    Exception ignored in:    ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^  ^
  ^  ^^Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^ ^    ^ self._shutdown_workers()^^ 
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ ^    ^ ^^if w.is_alive():^
^^
^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^     ^ ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^ 
^  ^^^  ^^  ^^^ ^^^^ ^  ^^ ^^^^^ ^^ ^^^ ^^^^^^^^^^^^^^^^^^^^^^^
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^

^^ AssertionError^^ : ^^can only test a child process
 ^^ ^^ ^^ ^^ Exception ignored in: ^
 ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^ 
AssertionError^ Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
: ^ can only test a child process^^    
self._shutdown_workers()^^^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^Exception ignored in: ^^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>    ^
if w.is_alive():^^
Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 
 ^    AssertionError^ self._shutdown_workers()
^:    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
can only test a child process^^     ^ 
if w.is_alive():^ ^^
^Exception ignored in:  ^^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^ ^^^Traceback (most recent call last):
 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^ ^^     ^ ^^^^self._shutdown_workers()^^^^
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^    ^if w.is_alive():^^^^^

^
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^     ^AssertionError ^ : assert self._parent_pid == os.getpid(), 'can only test a child process' 
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
can only test a child process
  
      ^^ assert self._parent_pid == os.getpid(), 'can only test a child process'
 Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  ^ 
  ^  Traceback (most recent call last):
^     File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      ^   self._shutdown_workers() ^
^ ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ ^^    ^^if w.is_alive():^^^
^^^^ ^
^^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^     ^^ assert self._parent_pid == os.getpid(), 'can only test a child process'^^
 ^^  ^ ^ ^^^ ^^ ^^^ ^^^ ^^^ ^^ ^^^^^  ^^^^^^^^ ^^^^^^^^
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^    ^^^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^
^^ ^^^^ ^^^^ ^
^^ ^AssertionError
:  ^ AssertionError^: can only test a child process 
^can only test a child process 
^Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
^ ^Exception ignored in: Traceback (most recent call last):
 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^    ^
self._shutdown_workers()^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^if w.is_alive():^    
^^^ self._shutdown_workers()^^ ^
^ ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^    ^^ if w.is_alive(): 

^ ^ AssertionError^ ^:  ^^can only test a child process ^
 ^^^^ Exception ignored in: ^ ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^^^
^Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^^    ^self._shutdown_workers()^^^
^^
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^    ^if w.is_alive():^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^
^^^  ^^  
^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
      ^assert self._parent_pid == os.getpid(), 'can only test a child process'   

     AssertionError^  : ^ can only test a child process 
  ^  ^Exception ignored in:  ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^^^ 
^^^ Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^^^ ^^    ^^^^self._shutdown_workers()

^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^    ^^if w.is_alive():assert self._parent_pid == os.getpid(), 'can only test a child process'^
^ ^^
 ^^  ^ ^ ^  ^^   ^  ^^  ^^^^ ^^ ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive

^^AssertionError    ^^: ^^^assert self._parent_pid == os.getpid(), 'can only test a child process'can only test a child process^^
^
^^ ^Exception ignored in: ^ ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 
^
^AssertionError Traceback (most recent call last):
:  can only test a child process  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^ 
^     self._shutdown_workers()^
 ^Exception ignored in:  ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ ^
    ^if w.is_alive(): Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

^^^     
self._shutdown_workers() ^AssertionError^ 
:  ^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
can only test a child process
 ^    ^ ^^if w.is_alive():^Exception ignored in: ^
^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 
^ Traceback (most recent call last):
 ^^ ^^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^     ^^self._shutdown_workers() ^^^^
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^^^    ^^if w.is_alive():
^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^^     ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^ ^
 ^^ ^ ^ ^ ^
     File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^ ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^
 ^^  ^^ 
  ^  AssertionError^  : ^  can only test a child process^ 
^^^^^  ^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
Exception ignored in: ^  ^    ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^assert self._parent_pid == os.getpid(), 'can only test a child process'
^^Traceback (most recent call last):
^
^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^ ^^     ^^^self._shutdown_workers()^^ 
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ ^    ^^ if w.is_alive():^^ 
^^^  ^  ^^^   ^^^  ^^ ^^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^^^AssertionError^^^: ^^^can only test a child process^

^^AssertionError^Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
^: 
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^can only test a child process      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^self._shutdown_workers()
^Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^Traceback (most recent call last):
    ^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
if w.is_alive():^ 
^      ^ self._shutdown_workers()^ 
 ^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
       ^  ^if w.is_alive(): ^
  
  ^AssertionError^ : ^^^can only test a child process 
^^ ^^Exception ignored in:  ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
 ^^Traceback (most recent call last):
 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    ^^^self._shutdown_workers()^^^
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^
^    ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^if w.is_alive():    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^^ 
 ^ ^^ ^^ ^ ^ ^ ^
^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
      ^assert self._parent_pid == os.getpid(), 'can only test a child process'  ^^ 
 ^^  ^^^  ^^^ ^ ^^
^ AssertionError^^ : ^^can only test a child process
^ ^^^ ^^^Exception ignored in:  ^^ ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
 ^^ Traceback (most recent call last):
^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^        ^^assert self._parent_pid == os.getpid(), 'can only test a child process'self._shutdown_workers()^

^^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ ^     ^^ if w.is_alive():^^ ^
^   ^^   ^ ^ ^^  ^^^ ^^ ^^^^ 
^^^^AssertionError^^^: ^^can only test a child process^^^
^^^^Exception ignored in: ^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^^
^^^Traceback (most recent call last):
^^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    ^^^self._shutdown_workers()^
^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    ^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'if w.is_alive():^^

^^   
 ^ AssertionError ^:  ^ can only test a child process ^ 
   ^ ^^Exception ignored in:  ^^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^ 
^^Traceback (most recent call last):
 ^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

AssertionError^^: ^    ^can only test a child process^^self._shutdown_workers()
^^^^
^^Exception ignored in:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>    

if w.is_alive():  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^Traceback (most recent call last):

    ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 assert self._parent_pid == os.getpid(), 'can only test a child process'^
    ^  self._shutdown_workers()^ ^  ^^
^    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^ ^     if w.is_alive(): ^ ^ 
^ ^^  ^ ^  ^  ^^^ ^ ^^^ ^^ ^^^^^^^^^^^^^
^^^AssertionError^^^: 
^can only test a child process^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive

^^    ^^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^Exception ignored in: ^
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^ 
^
^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^Traceback (most recent call last):
 ^       File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^ assert self._parent_pid == os.getpid(), 'can only test a child process'    ^ 
self._shutdown_workers()^ ^ 
^    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^  ^       ^^ ^if w.is_alive(): ^ 
^^^ ^  ^^^ ^^^ ^^ ^^ 
 ^AssertionError ^^: ^^^^^can only test a child process^^
^^Exception ignored in: ^^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^^^^^^
^Traceback (most recent call last):
^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    ^^self._shutdown_workers()^^^^^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^^    ^if w.is_alive():^^
^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^ ^    ^ ^ ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^ 
^  ^^  ^ ^
 ^ ^AssertionError^ : ^
can only test a child processAssertionError^ : ^
can only test a child process 
^ ^Exception ignored in:   ^Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 
^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^

^    ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
Traceback (most recent call last):
^self._shutdown_workers()^      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

^assert self._parent_pid == os.getpid(), 'can only test a child process'    ^
self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^
     ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 if w.is_alive():^
 ^     ^ if w.is_alive(): ^  
^   ^   ^  ^   ^^  ^ ^^ ^ ^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^AssertionError^: ^^    
assert self._parent_pid == os.getpid(), 'can only test a child process'^can only test a child process  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive

^     
^ ^assert self._parent_pid == os.getpid(), 'can only test a child process' Exception ignored in: 
^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 
 ^ Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^       ^self._shutdown_workers() 
  ^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  ^      ^^ if w.is_alive():^^
 ^^ ^   ^^^ ^^^^^^ ^^^ ^ ^^^^
^^^^^^^AssertionError^^^^: ^^^can only test a child process
^^^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^
^^ ^^^  ^^^ ^^ ^ ^ ^ ^ ^
^ AssertionError ^^: ^^^
^can only test a child process^AssertionError
^: ^can only test a child process^
^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Epoch 10/15  train=0.0021  val=0.0020  lr=1.42e-04  *
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

    Traceback (most recent call last):
if w.is_alive():  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

      self._shutdown_workers() 
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
       if w.is_alive():^
^ ^^ ^ ^^  ^ ^^ ^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^  ^ ^ ^ ^ ^ 
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
      assert self._parent_pid == os.getpid(), 'can only test a child process' 
^  ^  ^ ^  ^  ^ ^ ^^^Exception ignored in: ^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^^^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    ^^if w.is_alive():^^
^ ^ ^^ ^^ ^ ^^^ ^ ^^^^^^^^^^^^^^^^^^^^^^
^^Exception ignored in: ^AssertionError^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>: ^^^can only test a child processTraceback (most recent call last):

^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^Exception ignored in:     
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>self._shutdown_workers()
AssertionError

Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
:       File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    can only test a child process    assert self._parent_pid == os.getpid(), 'can only test a child process'if w.is_alive():

self._shutdown_workers()
 Exception ignored in: 
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>   
      Traceback (most recent call last):
 if w.is_alive(): 
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
               self._shutdown_workers()^  ^  ^
  ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    ^^^^^if w.is_alive():^^^
^ ^^^^^ ^^^ ^^^^
^ ^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^     ^ ^^assert self._parent_pid == os.getpid(), 'can only test a child process' ^^
^
^ ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^     ^^ ^assert self._parent_pid == os.getpid(), 'can only test a child process' ^^ 
^^  ^^^^  ^^  ^^  ^ ^  ^ ^^
 ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^     ^^ assert self._parent_pid == os.getpid(), 'can only test a child process'^ 
^^ ^ ^^^  ^ ^ ^^ ^^ ^^ ^ ^^^ 
AssertionError^ ^^: ^^^^^^can only test a child process^^^
^^^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^^^
^Traceback (most recent call last):
^^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    ^^^^self._shutdown_workers()^^^^^^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():^^^^^
^^^^ ^^^^^ ^^ ^^^^ 
^^ ^^AssertionError^ ^^:  
^AssertionErrorcan only test a child process^^: ^
can only test a child process^
^Exception ignored in: ^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^Exception ignored in: ^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
        AssertionError^self._shutdown_workers(): self._shutdown_workers()^can only test a child process

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

    if w.is_alive():      File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive

Exception ignored in:     if w.is_alive(): <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
assert self._parent_pid == os.getpid(), 'can only test a child process'  

 Traceback (most recent call last):
       File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
              ^ ^^ self._shutdown_workers()^ ^
^^ ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^ ^^^    ^^^^^^if w.is_alive():^^^^^^
^^ ^^^^ ^^
 ^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^assert self._parent_pid == os.getpid(), 'can only test a child process' ^
     ^  assert self._parent_pid == os.getpid(), 'can only test a child process'^^ 
^^ ^ ^^  ^^  ^^ ^ ^^  ^^  ^  ^ ^^^  ^^^ ^^^^ 
^^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^
^^^^ ^^^
^ ^^ ^^AssertionError^^  : ^^^can only test a child process ^^ 
^^  ^^ ^ ^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^^^
^^^Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^^    ^^^^^self._shutdown_workers()^^^
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    ^^if w.is_alive():^
^^^^^^ ^^ ^^^ ^^
 ^ AssertionError^^ : ^ 
can only test a child process^^AssertionError^
^: ^^Exception ignored in: ^can only test a child process^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
^^^^
^^Traceback (most recent call last):
^Exception ignored in: ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^    
^^Traceback (most recent call last):
self._shutdown_workers()^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
self._shutdown_workers()    ^    
^if w.is_alive():assert self._parent_pid == os.getpid(), 'can only test a child process'

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

AssertionError :   can only test a child process    
     if w.is_alive(): Exception ignored in: 
 <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 
   Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  ^    ^  ^ self._shutdown_workers()  
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  ^^^^    ^if w.is_alive():^^^
^^^ ^^^ ^^ ^^^ ^^^^ 
^ ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^     ^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^
^^ ^ ^^^
^^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^ ^assert self._parent_pid == os.getpid(), 'can only test a child process'^ ^
^ ^ ^ ^^^  ^ ^ ^  ^ ^  
 ^ ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^^    ^^ ^assert self._parent_pid == os.getpid(), 'can only test a child process'^ ^
^^^^ ^^ ^^ 
^ AssertionError^ : ^^ can only test a child process^^
 ^^^  Exception ignored in: ^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^^
 ^^Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    ^^self._shutdown_workers()^^^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^^    ^^^if w.is_alive():^^
^^^^^ ^^ ^^^^  ^^^^^^ ^ ^^ ^^^^^^^^^
^^AssertionError^^^^^: ^^^^can only test a child process^^
^^^^^^^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
^

AssertionError^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
: Traceback (most recent call last):
^    can only test a child process  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError    
: self._shutdown_workers() can only test a child processException ignored in: 
 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 
 Traceback (most recent call last):
    Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>if w.is_alive():  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__


  Traceback (most recent call last):
       File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
self._shutdown_workers()
          File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 self._shutdown_workers()      
 if w.is_alive():^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    
  ^ ^if w.is_alive():^^ 
 ^^   ^^  ^ ^  ^^^ ^^^ ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^ ^     ^assert self._parent_pid == os.getpid(), 'can only test a child process'
 ^ ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive

  ^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'  ^ 
 ^    ^ 
   AssertionError    : ^ can only test a child process^  
^   ^^ ^Exception ignored in:  ^^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^ 
^^Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    ^^self._shutdown_workers()^
^^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^    ^^^if w.is_alive():^^^
^^ ^^^^ ^^^ ^ ^^^ ^^^ ^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^AssertionError^^^: ^^^can only test a child process^^^

^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>    ^^^
assert self._parent_pid == os.getpid(), 'can only test a child process'


Traceback (most recent call last):
AssertionError   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
AssertionError : :      can only test a child process can only test a child processself._shutdown_workers() 


  Exception ignored in:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00><function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 

    if w.is_alive():Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

         self._shutdown_workers()^ 
 ^self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 
    ^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
if w.is_alive(): 
^     ^if w.is_alive():  ^
^^  ^^ ^^^  ^^  ^ ^^ ^ ^ ^^^ ^^ ^^^^^^^^^^^^
^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^    assert self._parent_pid == os.getpid(), 'can only test a child process'^^^^^
^^^ ^^^ ^^^^  ^^^
 
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
     ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 assert self._parent_pid == os.getpid(), 'can only test a child process'    ^
  assert self._parent_pid == os.getpid(), 'can only test a child process'^   
^    ^  
^  AssertionError^  : ^  ^can only test a child process  
^  ^^  Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^
^ ^Traceback (most recent call last):
^^ ^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    ^^self._shutdown_workers()^^^
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^^    ^^if w.is_alive():^
^^ ^^ ^^^^ ^^^  ^^^ ^^^ ^^^^^^^^^^^^^^^^^^^^^^^^
^^^AssertionError^^: ^^^^^^can only test a child process^^^
^^^^^^^
^Exception ignored in: ^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^    

assert self._parent_pid == os.getpid(), 'can only test a child process'Traceback (most recent call last):
AssertionError^
:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^ can only test a child process^     
self._shutdown_workers()
 Exception ignored in: AssertionError
 <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 
can only test a child process    
Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
if w.is_alive():    
 Exception ignored in: self._shutdown_workers()<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>    

Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^     ^    self._shutdown_workers()if w.is_alive(): ^
^ 
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^      ^if w.is_alive(): ^ ^
 ^^ ^  ^^   ^^^ ^^^^^ ^^ ^^^^^^^^^^^^^^^^^^
^^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^^^ ^^^ ^^^ 
^ ^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^ 
    ^ assert self._parent_pid == os.getpid(), 'can only test a child process'  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^ 
    ^ assert self._parent_pid == os.getpid(), 'can only test a child process' ^
  
    AssertionError   :   ^can only test a child process ^ 
  ^  ^Exception ignored in: ^  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^  
 ^ Traceback (most recent call last):
^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    ^self._shutdown_workers()^^^
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^    ^^^^if w.is_alive():^
^^^ ^^^^ ^^ ^^^ ^^^^ ^^^  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^AssertionError
^^: ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    can only test a child process^assert self._parent_pid == os.getpid(), 'can only test a child process'
^^^^
Exception ignored in:  ^^^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^

 
Traceback (most recent call last):
AssertionError  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 : AssertionErrorcan only test a child process :     can only test a child process
  self._shutdown_workers()

 Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 Exception ignored in:     Traceback (most recent call last):
if w.is_alive(): <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__


^ Traceback (most recent call last):
       File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^ self._shutdown_workers()^ 
     ^self._shutdown_workers()   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    ^    ^^if w.is_alive():^if w.is_alive():^

^ ^^  ^  ^ ^^  ^^  ^ ^   ^^^^^^^^^^^^^
^^^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^^^  ^^^^ ^^^ ^^^^
 ^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^
       File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^ assert self._parent_pid == os.getpid(), 'can only test a child process'     ^
assert self._parent_pid == os.getpid(), 'can only test a child process'^ 
   ^ ^ ^^^  ^ 
 ^AssertionError  ^:   ^ can only test a child process ^ 
 ^  Exception ignored in: ^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^
  ^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^^    ^self._shutdown_workers()^^^
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^    ^^^^^^^if w.is_alive():^
^^^ ^^^^ ^^^^ ^^^ ^^ ^^^ ^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^AssertionError^^^^: ^^can only test a child process^^
^^^^^^
^Exception ignored in: ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>    
^
Traceback (most recent call last):
assert self._parent_pid == os.getpid(), 'can only test a child process'^AssertionError  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

:     
can only test a child processself._shutdown_workers() AssertionError
 
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  : Exception ignored in:     can only test a child process <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
if w.is_alive(): 

 Exception ignored in: Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

  Traceback (most recent call last):
        self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
     ^self._shutdown_workers()     ^
if w.is_alive():  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^
^^     ^ ^if w.is_alive():^^ 
^^ ^^^  ^^   ^^   ^^ ^^ ^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^^    ^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^
^^^^ ^^^ ^^^^ 
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^
^    ^ assert self._parent_pid == os.getpid(), 'can only test a child process'  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^
     ^  
assert self._parent_pid == os.getpid(), 'can only test a child process'  AssertionError:   
  can only test a child process    
 ^  ^  ^  ^Exception ignored in:   ^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^ ^
^ Traceback (most recent call last):
^^ ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^     ^ self._shutdown_workers()^^^
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^^^if w.is_alive():^^
^^^^ ^^^^^ ^^^ ^ ^^^^ ^ ^^^^^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^AssertionError^^: ^^^can only test a child process^^
AssertionError

: ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
can only test a child processException ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>    ^assert self._parent_pid == os.getpid(), 'can only test a child process'

^
Traceback (most recent call last):
 Exception ignored in:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^     self._shutdown_workers()^
 
 ^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Traceback (most recent call last):
^       File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^
if w.is_alive():      
AssertionErrorself._shutdown_workers() :  
 ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
can only test a child process  ^
    ^ if w.is_alive():^ 
Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  ^ 
^ Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^     ^self._shutdown_workers()^^ 
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^^     ^^if w.is_alive():^^^^
^^ ^^^^^ ^ ^^^^ ^^^ ^^^ 
^^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^^
^^^ ^^^^ ^^
^ ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^ ^
     ^AssertionError ^assert self._parent_pid == os.getpid(), 'can only test a child process' ^
 : ^ can only test a child process  ^ 
  
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^     assert self._parent_pid == os.getpid(), 'can only test a child process'^
 ^
Traceback (most recent call last):
 ^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      ^^ self._shutdown_workers()^^ ^ 
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^ ^     ^^if w.is_alive():
^ ^^^  ^^  ^^  ^^  ^^ ^^^ ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^    ^^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^
^^
 
^AssertionError^AssertionError : :  ^can only test a child process ^can only test a child process 

^ ^Exception ignored in: ^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
 Exception ignored in: ^Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^
     Traceback (most recent call last):
^self._shutdown_workers()   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__


^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^AssertionError    : ^can only test a child process    ^self._shutdown_workers()if w.is_alive():
^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^
^^    ^Exception ignored in:   <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>if w.is_alive(): ^
 
^ ^Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
       ^^self._shutdown_workers() 
^^  ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^     if w.is_alive():^^^
^ ^^ ^^^ ^^^ ^ ^ ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^
^    AssertionError    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'assert self._parent_pid == os.getpid(), 'can only test a child process': 

  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 can only test a child process
     assert self._parent_pid == os.getpid(), 'can only test a child process' 

    Exception ignored in:      <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>  
    Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
         self._shutdown_workers()  
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  ^      ^if w.is_alive():^
^^^^ ^^^ ^^^ ^^^ ^ ^^^ ^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^^^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^^
^^^ ^^^ ^^^^ ^^^^ ^^ ^^^ 
 ^^AssertionError^^:  ^
 can only test a child processAssertionError
 :  AssertionError
can only test a child process^: 
^can only test a child process^
^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^^if w.is_alive():
^^  ^^ ^ ^ ^ ^ ^^^^^^^^^^^^^^^
^AssertionError^^: ^can only test a child process^

  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Epoch 11/15  train=0.0020  val=0.0020  lr=9.41e-05  *
Epoch 12/15  train=0.0020  val=0.0020  lr=5.45e-05  
Epoch 13/15  train=0.0020  val=0.0019  lr=2.48e-05  *
Epoch 14/15  train=0.0020  val=0.0019  lr=6.27e-06  *
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 
 Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
       self._shutdown_workers()^^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^if w.is_alive():^^^
^  ^ ^  ^ 
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^ ^^ ^ ^^ ^  ^ ^ 
    File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
     ^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^
 ^^ ^ ^^  ^^ ^^ ^ ^^  ^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^AssertionError^^: can only test a child process^
^^Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^if w.is_alive():^
AssertionError
:  can only test a child process  
  Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 
 Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^self._shutdown_workers()^^
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^^if w.is_alive():^
   ^ 
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
      ^assert self._parent_pid == os.getpid(), 'can only test a child process'
^ ^ ^^ ^^ ^ ^ ^ ^ ^ 
    File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^^  ^  ^  ^ ^ ^^ ^ ^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^AssertionError^: ^can only test a child process^
^^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
^^Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
AssertionError    : self._shutdown_workers()can only test a child process
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

    if w.is_alive():Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>

Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      self._shutdown_workers() 
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    ^if w.is_alive():^^
^ ^ ^ ^ ^ ^ ^^ ^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^ ^ ^  ^ ^   ^ ^ 
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    assert self._parent_pid == os.getpid(), 'can only test a child process'^^
^   ^^  ^^ ^ ^ ^ ^ ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^AssertionError^: ^^can only test a child process
^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^if w.is_alive():
^ ^
 AssertionError  :  can only test a child process 
 ^Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
^Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    
if w.is_alive():  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive

      assert self._parent_pid == os.getpid(), 'can only test a child process'
            ^ ^^ ^ ^^ ^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    assert self._parent_pid == os.getpid(), 'can only test a child process'^Exception ignored in: 
^  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
^  Traceback (most recent call last):
^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^     ^self._shutdown_workers() ^
 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^    if w.is_alive(): ^
 ^^ ^^^ ^^^ ^^  ^^^^  ^^^^^^
^AssertionError^: ^can only test a child process^^
^^^^^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^^^^Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^^^self._shutdown_workers()^

  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^assert self._parent_pid == os.getpid(), 'can only test a child process'
    ^  if w.is_alive():^  
^ ^  ^ ^   ^  
  AssertionError  : ^ ^can only test a child process^^^^
^^^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^^self._shutdown_workers()^
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    
^if w.is_alive():  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^
^    assert self._parent_pid == os.getpid(), 'can only test a child process'^ ^ 
 ^  ^    ^   ^   ^^ ^ ^^^ ^^^^^^^^^^^^^^^^^^^^
^^AssertionError^^: ^
^can only test a child process^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>assert self._parent_pid == os.getpid(), 'can only test a child process'^
^Traceback (most recent call last):
^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^      ^ self._shutdown_workers()^ ^
^ ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
      ^ ^ ^if w.is_alive():^ 
  ^^^ ^^^ ^^^
 ^AssertionError ^ : ^can only test a child process ^
^^^^Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    ^^self._shutdown_workers()
^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^^^    ^if w.is_alive():
^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^     ^ assert self._parent_pid == os.getpid(), 'can only test a child process' ^
  ^  ^ ^ ^Exception ignored in: ^^ <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
^ Traceback (most recent call last):
^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    
^self._shutdown_workers()AssertionError ^
 : ^ ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
     can only test a child process^ if w.is_alive():

  ^ ^^Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
 ^ 
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 ^Traceback (most recent call last):
       File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^    
^self._shutdown_workers() ^^
 ^^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^     ^^ ^^if w.is_alive():^ ^^
 ^^  ^^^ ^^  
 ^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
      ^^assert self._parent_pid == os.getpid(), 'can only test a child process' ^^ 
 ^^^ ^^ ^^^^^ ^^^^ ^^ ^^^^ ^^ ^^ ^^ ^^^ ^^^^
^^^
^AssertionError^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
: ^^^    can only test a child process^^^assert self._parent_pid == os.getpid(), 'can only test a child process'
^
^^^^ Exception ignored in: ^^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^
^^ Traceback (most recent call last):
^ ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^     ^ ^self._shutdown_workers() ^^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^^     ^if w.is_alive():^^^
 ^ ^ 
  ^AssertionError^^ : can only test a child process^ ^
^ ^ ^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^^^^Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^^    ^^self._shutdown_workers()^
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^
^    ^AssertionError^if w.is_alive():^
^: ^^can only test a child process ^

 ^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
Exception ignored in: ^    <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> ^
 assert self._parent_pid == os.getpid(), 'can only test a child process'^Traceback (most recent call last):
 ^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^ ^     ^^self._shutdown_workers() ^^
^ ^   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^ ^     ^ ^if w.is_alive():^^
^ ^^ ^ ^^  
^ ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^     ^
^ AssertionError^ assert self._parent_pid == os.getpid(), 'can only test a child process':  ^^can only test a child process
^^
 ^ ^^ ^^Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^Traceback (most recent call last):

^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^^     ^^self._shutdown_workers()
^ ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^     ^ ^if w.is_alive(): ^
^  
 ^^   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^      ^ ^assert self._parent_pid == os.getpid(), 'can only test a child process'^ ^^^
^^^^ ^^^^^^ ^^^^ ^^^^^ ^^^ ^^^ ^^
^ ^ ^AssertionError: ^^ 
can only test a child process^ 
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
     ^assert self._parent_pid == os.getpid(), 'can only test a child process'^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^^ 
Traceback (most recent call last):
^ ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
 ^     ^self._shutdown_workers()^ 
^ ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  ^^     ^^ if w.is_alive():^
 ^^ ^^^ ^^ ^^ ^^^^^^ ^^ ^^ ^
^AssertionError^^: ^^can only test a child process^^
^^^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
^^^^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^^    ^^self._shutdown_workers()^^^

^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
AssertionError^    ^: ^if w.is_alive():can only test a child process^^^

^^ ^ ^
 Exception ignored in:   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^     ^
Traceback (most recent call last):
^ assert self._parent_pid == os.getpid(), 'can only test a child process' ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

 ^ ^    self._shutdown_workers()^ ^^
 ^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
     ^^ ^ if w.is_alive():^
 
 ^ AssertionError ^ : ^ can only test a child process^  ^ 
 ^^Exception ignored in:  ^^ ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^
^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^        ^assert self._parent_pid == os.getpid(), 'can only test a child process'^self._shutdown_workers()^^

   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^    ^ ^^ if w.is_alive():^
^  ^^^  ^^  
  ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
   ^ ^       ^^assert self._parent_pid == os.getpid(), 'can only test a child process'^^^
^^^ ^ ^^ ^^ ^^ ^^^^ ^^^^^^ ^^ ^^^ ^^^ ^^^^ ^
^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^AssertionError    ^assert self._parent_pid == os.getpid(), 'can only test a child process': ^^
 can only test a child process^ ^
^ ^ ^^ ^^ Exception ignored in:  ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^ ^
^ ^ Traceback (most recent call last):
^^ ^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^^^^^^self._shutdown_workers()^^
^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^^    
^^if w.is_alive():AssertionError
^^:  ^^can only test a child process ^
^^^ ^^ ^^ ^^^ ^^ ^^^^^^^^^^^^^^
^^AssertionError^^: ^^can only test a child process^^^
^^^^Exception ignored in: ^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>


  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
Traceback (most recent call last):
AssertionError  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    : assert self._parent_pid == os.getpid(), 'can only test a child process'    can only test a child process
self._shutdown_workers() 
 
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
     if w.is_alive(): 
            ^^ ^^^^^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^ ^  ^ ^ ^ ^^^ ^ ^ ^^^ ^ ^^^^^
AssertionError^: can only test a child process^
^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>^^
Traceback (most recent call last):
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    if w.is_alive():^
^  ^ ^ ^^ ^ ^^ ^^^^^^^^^^^^
AssertionError^: ^^can only test a child process^^
^
Exception ignored in:   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>    assert self._parent_pid == os.getpid(), 'can only test a child process'

Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
     self._shutdown_workers()
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
       if w.is_alive():
          ^ ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'
^ ^  ^ ^ 
AssertionError  : can only test a child process
  Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    self._shutdown_workers()^
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^^if w.is_alive():
^  ^  ^^^  ^^ ^^^^^^^^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError
:  can only test a child process
  Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00> 
 Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      self._shutdown_workers() 
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
      if w.is_alive():^
^ ^ ^ ^ ^^ ^^ ^ ^^^^^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^ ^ ^  ^  ^^ ^
  AssertionError :  ^can only test a child process^^
^^Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
^^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    ^if w.is_alive():^
^ ^  ^ ^ ^ ^ ^^^^^^^^^^^^^^
^AssertionError: can only test a child process^^
^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>

Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
        self._shutdown_workers()assert self._parent_pid == os.getpid(), 'can only test a child process'
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

     if w.is_alive():
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    assert self._parent_pid == os.getpid(), 'can only test a child process'^^
^ ^ ^ ^ ^^ ^  ^  ^ ^ ^^
^AssertionError: ^can only test a child process
^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b23f0e95d00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Epoch 15/15  train=0.0020  val=0.0019  lr=0.00e+00  

Best val loss : 0.0019
Checkpoint    : /kaggle/working/bird_ast_model.pth
Target columns saved : /kaggle/working/target_columns.json

Done.
Training plot saved : /kaggle/working/loss_plot.png
